# Mask Models

Once we have the data, we can build some models to test them and see how they perform versus one another. Something we care about is the performance of the model (accuracy) of predicting whether or not something is a bulk layer or not.


In [3]:
%pip install PyWavelets
%pip install tqdm
%pip install autogluon

In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm import tqdm
import pywt
import autogluon.core as ag
from autogluon.tabular import TabularDataset, TabularPredictor

In [11]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split

# ── 1. Load your preprocessed CSV ──────────────────────────────────────────
feature_function_name = "wavelet_features"  # or "simple_statistics"
resolution = 4                             # whichever resolution you used

df = pd.read_csv(f"./{feature_function_name}_resolution_{resolution}.csv")

# ── 2. Ensure target is binary ─────────────────────────────────────────────
# Inspect unique values first
print("Unique 'bulk' values:", df['bulk'].unique())

# If bulk is numeric, binarize it (adjust threshold as needed)
# e.g. if 0 = no error, anything else = error
df['target'] = (df['id2_binary'] != 0).astype(int)

# ── 3. Drop columns that would leak identity info ──────────────────────────
# 'file' and 'bulk' are not features — drop them
# Keep: scan_number, subsegment_index, mean_laser_current, all wavelet/stat cols
drop_cols = ['file', 'bulk', 'id2', 'id2_binary']
df = df.drop(columns=drop_cols)

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")

# ── 4. Train/test split — split by file to avoid data leakage ──────────────
# (if you kept 'file' col, group-split here instead)
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['target'])

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# ── 5. Convert to AutoGluon Dataset ───────────────────────────────────────
train_data = TabularDataset(train_df)
test_data  = TabularDataset(test_df)

# ── 6. Train ───────────────────────────────────────────────────────────────
predictor = TabularPredictor(
    label='target',
    problem_type='binary',
    eval_metric='roc_auc',      # good default for imbalanced binary classification
    path='./autogluon_models'   # saves models here for reuse
).fit(
    train_data,
    time_limit=600,             # seconds — increase for better results
    presets='best_quality',     # or 'medium_quality' for faster runs
    verbosity=2
)

# ── 7. Evaluate ────────────────────────────────────────────────────────────
leaderboard = predictor.leaderboard(test_data, silent=False)
print(leaderboard)

# ── 8. Detailed performance metrics ───────────────────────────────────────
performance = predictor.evaluate(test_data)
print("\nPerformance metrics:", performance)

# ── 9. Feature importance ──────────────────────────────────────────────────
importance = predictor.feature_importance(test_data)
print("\nFeature importance:\n", importance)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       9.05 GB / 12.67 GB (71.4%)
Disk Space Avail:   72.20 GB / 107.72 GB (67.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` v

Unique 'bulk' values: [1. 0.]
Dataset shape: (60914, 10)
Target distribution:
target
0    56457
1     4457
Name: count, dtype: int64
Train: (42639, 10), Test: (18275, 10)


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0    LightGBMXT_BAG_L2       0.785851   0.778242     roc_auc        3.683208       4.552384  131.371789                 0.804529                2.077687          41.685233            2       True          4
1  WeightedEnsemble_L3       0.783436   0.781024     roc_auc        3.687888       4.563215  132.636100                 0.004679                0.010831           1.264312            3       True          5
2  WeightedEnsemble_L2       0.782261   0.780666     roc_auc        2.882833       2.510623   91.238948                 0.004154                0.035926           1.552393            2       True          3
3    LightGBMXT_BAG_L1       0.781496   0.779378     roc_auc        2.237051       1.718061   49.544845                 2.237051     

                     model  score_test  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      WeightedEnsemble_L3    0.786453   0.783931     roc_auc       12.383643       9.301036  342.935739                 0.004525                0.008746           1.400556            3       True          8
1      WeightedEnsemble_L2    0.785922   0.782589     roc_auc        7.717814       5.452565  178.406783                 0.003854                0.008914           1.110737            2       True          5
2        LightGBMXT_BAG_L1    0.785502   0.780129     roc_auc        4.069902       1.558467   49.477827                 4.069902                1.558467          49.477827            1       True          1
3        LightGBMXT_BAG_L2    0.785465   0.781864     roc_auc       11.710705       8.792853  299.454551                 1.925900                0.991589          45.65

Computing feature importance via permutation shuffling for 9 features using 5000 rows with 5 shuffle sets...



Performance metrics: {'roc_auc': np.float64(0.7864527791223798), 'accuracy': 0.9377291381668946, 'balanced_accuracy': np.float64(0.59715418624288), 'mcc': np.float64(0.37941513834895135), 'f1': 0.31774580335731417, 'precision': 0.8006042296072508, 'recall': 0.19820493642483172}


	157.93s	= Expected runtime (31.59s per shuffle set)
	159.94s	= Actual runtime (Completed 5 of 5 shuffle sets)



Feature importance:
                               importance    stddev   p_value  n  p99_high  \
wavelet_level_4_energy_ratio    0.136959  0.017080  0.000028  5  0.172127   
wavelet_level_5_energy_ratio    0.031476  0.010177  0.001147  5  0.052430   
wavelet_level_3_energy_ratio    0.022930  0.003847  0.000092  5  0.030851   
mean_laser_current              0.020617  0.003461  0.000092  5  0.027744   
scan_number                     0.020052  0.002611  0.000034  5  0.025429   
subsegment_index                0.016285  0.006914  0.003113  5  0.030521   
wavelet_level_2_energy_ratio    0.005186  0.002365  0.004014  5  0.010057   
wavelet_level_0_energy_ratio    0.000896  0.002018  0.188437  5  0.005052   
wavelet_level_1_energy_ratio   -0.000389  0.002025  0.655302  5  0.003779   

                               p99_low  
wavelet_level_4_energy_ratio  0.101792  
wavelet_level_5_energy_ratio  0.010523  
wavelet_level_3_energy_ratio  0.015010  
mean_laser_current            0.013491  
sc

In [14]:
from sklearn.model_selection import train_test_split

# ── 1. Load your preprocessed CSV ──────────────────────────────────────────
feature_function_name = "wavelet_features"
resolution = 4

df = pd.read_csv(f"./{feature_function_name}_resolution_{resolution}.csv")
df['target'] = (df['id2_binary'] != 0).astype(int)
df = df.drop(columns=['bulk', 'id2', 'id2_binary'])

# ── 2. Define masking function ─────────────────────────────────────────────
def mask_layer(df, fraction):
    """
    Keep only the first `fraction` of subsegments per (file, scan_number).
    e.g. fraction=0.2 keeps the first 20% of subsegments in each scan.
    """
    def keep_first_fraction(group):
        n_keep = max(1, int(np.ceil(len(group) * fraction)))
        return group.iloc[:n_keep]

    return (
        df.groupby(['file', 'scan_number'], group_keys=False)
          .apply(keep_first_fraction)
          .reset_index(drop=True)
    )

# ── 3. Run AutoGluon for each mask fraction ────────────────────────────────
fractions = [0.2, 0.3, 0.5, 0.7, 1]
results = {}  # stores performance metrics per fraction

for fraction in fractions:
    print(f"\n{'='*60}")
    print(f"  Training on first {int(fraction*100)}% of each layer")
    print(f"{'='*60}")

    # Apply mask
    masked_df = mask_layer(df, fraction)
    masked_df = masked_df.drop(columns=['file'])  # drop after masking

    print(f"  Masked dataset shape: {masked_df.shape}")
    print(f"  Target distribution:\n{masked_df['target'].value_counts()}")

    # Train/test split
    train_df, test_df = train_test_split(
        masked_df,
        test_size=0.3,
        random_state=42,
        stratify=masked_df['target']
    )

    train_data = TabularDataset(train_df)
    test_data  = TabularDataset(test_df)

    # Train
    predictor = TabularPredictor(
        label='target',
        problem_type='binary',
        eval_metric='roc_auc',
        path=f'./autogluon_models/fraction_{int(fraction*100)}pct'
    ).fit(
        train_data,
        time_limit=600,
        presets='best_quality',
        verbosity=1
    )

    # Evaluate
    performance = predictor.evaluate(test_data)
    leaderboard = predictor.leaderboard(test_data, silent=True)
    best_model  = leaderboard.iloc[0]  # top row = best model

    results[fraction] = {
        'performance': performance,
        'best_model_name': best_model['model'],
        'best_model_score': best_model['score_test'],
        'leaderboard': leaderboard
    }

    print(f"\n  ✅ Best model: {best_model['model']} | ROC-AUC: {best_model['score_test']:.4f}")

# ── 4. Summary table ───────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  SUMMARY")
print(f"{'='*60}")

summary = pd.DataFrame([
    {
        'fraction': f"{int(f*100)}%",
        'best_model': results[f]['best_model_name'],
        'roc_auc': results[f]['best_model_score'],
        **results[f]['performance']  # expands all metrics (accuracy, f1, etc.)
    }
    for f in fractions
])

print(summary.to_string(index=False))
summary.to_csv('./autogluon_models/summary.csv', index=False)
print("\nSummary saved to ./autogluon_models/summary.csv")


  Training on first 20% of each layer


/tmp/ipykernel_566/3003553159.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_first_fraction)


  Masked dataset shape: (35443, 10)
  Target distribution:
target
0    33091
1     2352
Name: count, dtype: int64

  ✅ Best model: WeightedEnsemble_L2 | ROC-AUC: 0.7714

  Training on first 30% of each layer


/tmp/ipykernel_566/3003553159.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_first_fraction)


  Masked dataset shape: (35644, 10)
  Target distribution:
target
0    33280
1     2364
Name: count, dtype: int64

  ✅ Best model: WeightedEnsemble_L2 | ROC-AUC: 0.7580

  Training on first 50% of each layer


...

  Masked dataset shape: (60713, 10)
  Target distribution:
target
0    56268
1     4445
Name: count, dtype: int64

  ✅ Best model: WeightedEnsemble_L2 | ROC-AUC: 0.7716

  Training on first 100% of each layer


/tmp/ipykernel_566/3003553159.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(keep_first_fraction)


  Masked dataset shape: (60914, 10)
  Target distribution:
target
0    56457
1     4457
Name: count, dtype: int64


	Not enough time to generate out-of-fold predictions for model. Estimated time required was 10.3s compared to 10s of available time.



  ✅ Best model: WeightedEnsemble_L3 | ROC-AUC: 0.7865

  SUMMARY
fraction          best_model  roc_auc  accuracy  balanced_accuracy      mcc       f1  precision   recall
     20% WeightedEnsemble_L2 0.770067  0.941973           0.580135 0.348840 0.271547   0.815603 0.162890
     30% WeightedEnsemble_L2 0.757349  0.941463           0.574257 0.335918 0.254762   0.816794 0.150917
     50% WeightedEnsemble_L2 0.776178  0.937583           0.572839 0.324076 0.250000   0.781250 0.148810
     70% WeightedEnsemble_L2 0.771585  0.936862           0.590367 0.365200 0.299635   0.798701 0.184408
    100% WeightedEnsemble_L3 0.786453  0.937729           0.597154 0.379415 0.317746   0.800604 0.198205

Summary saved to ./autogluon_models/summary.csv


We're gonna be so honest here. Jonathan's scared of installing autogluon and destroying his computer and he did it last year, so he did this on Google Collab.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
shutil.make_archive(
    base_name='/content/drive/MyDrive/autogluon_models',  # destination
    format='zip',
    root_dir='/content',
    base_dir='autogluon_models'                           # folder to zip
)